# Harris Hawks Optimization (HHO)

Ce notebook implémente l'algorithme d'optimisation **Harris Hawks Optimization (HHO)**.
L'algorithme est appliqué à un problème d'optimisation de conception de médicaments (Drug Design) utilisant des molécules sous forme de texte (SMILES), ainsi qu'à une fonction de test standard continue avec visualisation.

## Caractéristiques
- Typage statique (Type hints)
- Docstrings détaillés
- Visualisation de la convergence et de l'espace de recherche
- HHO continu (pour les fonctions mathématiques) et HHO discret (pour les séquences SMILES)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import random
from typing import Callable, Tuple, List

def levy_flight(dim: int) -> np.ndarray:
    """
    Génère un pas de vol de Lévy.
    
    Args:
        dim (int): La dimension de l'espace de recherche.
        
    Returns:
        np.ndarray: Un vecteur numpy représentant le pas de Lévy.
    """
    beta = 1.5
    sigma = (math.gamma(1 + beta) * math.sin(math.pi * beta / 2) / 
             (math.gamma((1 + beta) / 2) * beta * 2**((beta - 1) / 2)))**(1 / beta)
    u = np.random.randn(dim) * sigma
    v = np.random.randn(dim)
    step = u / np.abs(v)**(1 / beta)
    return step

In [ ]:
# def hho(
#     obj_func: Callable[[np.ndarray], float],
#     lb: np.ndarray,
#     ub: np.ndarray,
#     dim: int,
#     pop_size: int = 30,
#     max_iter: int = 100
# ) -> Tuple[np.ndarray, float, List[float], List[np.ndarray]]:
#     """
#     Implémentation de l'algorithme Harris Hawks Optimization (HHO) Continu.
#     """
#     # Initialisation de la population
#     X = np.random.uniform(0, 1, (pop_size, dim)) * (ub - lb) + lb
    
#     rabbit_location = np.zeros(dim)
#     rabbit_energy = float("inf")
    
#     convergence_curve = []
#     history = []
    
#     for t in range(max_iter):
#         for i in range(pop_size):
#             X[i, :] = np.clip(X[i, :], lb, ub)
#             fitness = obj_func(X[i, :])
#             if fitness < rabbit_energy:
#                 rabbit_energy = fitness
#                 rabbit_location = X[i, :].copy()
                
#         E1 = 2 * (1 - (t / max_iter)) # Facteur d'énergie d'échappement
        
#         for i in range(pop_size):
#             E0 = 2 * np.random.rand() - 1
#             E = E1 * E0
            
#             if abs(E) >= 1:
#                 q = np.random.rand()
#                 rand_hawk_index = np.random.randint(0, pop_size)
#                 X_rand = X[rand_hawk_index, :]
                
#                 if q >= 0.5:
#                     X[i, :] = X_rand - np.random.rand() * abs(X_rand - 2 * np.random.rand() * X[i, :])
#                 else:
#                     X_mean = np.mean(X, axis=0)
#                     X[i, :] = (rabbit_location - X_mean) - np.random.rand() * (lb + np.random.rand() * (ub - lb))
#             else:
#                 r = np.random.rand()
                
#                 if r >= 0.5 and abs(E) >= 0.5:
#                     J = 2 * (1 - np.random.rand())
#                     X[i, :] = (rabbit_location - X[i, :]) - E * abs(J * rabbit_location - X[i, :])
                    
#                 elif r >= 0.5 and abs(E) < 0.5:
#                     X[i, :] = rabbit_location - E * abs(rabbit_location - X[i, :])
                    
#                 elif r < 0.5 and abs(E) >= 0.5:
#                     J = 2 * (1 - np.random.rand())
#                     Y = rabbit_location - E * abs(J * rabbit_location - X[i, :])
#                     if obj_func(Y) < obj_func(X[i, :]):
#                         X[i, :] = Y
#                     else:
#                         Z = Y + np.random.rand(dim) * levy_flight(dim)
#                         if obj_func(Z) < obj_func(X[i, :]):
#                             X[i, :] = Z
                            
#                 elif r < 0.5 and abs(E) < 0.5:
#                     J = 2 * (1 - np.random.rand())
#                     X_mean = np.mean(X, axis=0)
#                     Y = rabbit_location - E * abs(J * rabbit_location - X_mean)
#                     if obj_func(Y) < obj_func(X[i, :]):
#                         X[i, :] = Y
#                     else:
#                         Z = Y + np.random.rand(dim) * levy_flight(dim)
#                         if obj_func(Z) < obj_func(X[i, :]):
#                             X[i, :] = Z

#         convergence_curve.append(rabbit_energy)
#         history.append(rabbit_location.copy())
        
#     return rabbit_location, rabbit_energy, convergence_curve, history

In [ ]:
# --- Cas 1 : Optimisation de Conception de Médicaments avec molécules réelles (SMILES) ---
from rdkit import Chem
from rdkit.Chem import Descriptors
from IPython.display import display


# lazyPredict 
def smiles_fitness(smiles: str) -> float:
    """
    Évalue une molécule SMILES. 
    On veut minimiser la différence de poids moléculaire par rapport à 300,
    et minimiser le nombre de donneurs et d'accepteurs d'hydrogène.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return float('inf') # Pire score si la molécule est invalide
        
    mw = Descriptors.MolWt(mol)
    donors = Descriptors.NumHDonors(mol)
    acceptors = Descriptors.NumHAcceptors(mol)

    score = abs(mw - 300) + donors + acceptors
    return score

def mutate_smiles(smiles: str, intensity: float) -> str:
    """
    Mute une chaîne SMILES. L'intensité détermine le nombre de mutations.
    """
    chars = list(smiles)
    if len(chars) == 0: return "C"
    
    num_mutations = max(1, int(len(chars) * intensity))
    
    for _ in range(num_mutations):
        if len(chars) == 0: break
        i = random.randint(0, len(chars) - 1)
        action = random.choice(['sub', 'add', 'del'])
        
        if action == 'sub':
            chars[i] = random.choice(["C", "O", "N", "F", "S", "Cl", "(", ")", "=", "#"])
        elif action == 'add':
            chars.insert(i, random.choice(["C", "O", "N", "(", ")", "="]))
        elif action == 'del' and len(chars) > 1:
            chars.pop(i)
            
    return "".join(chars)

def crossover_smiles(smiles1: str, smiles2: str) -> str:
    """
    Croisement simple entre deux chaînes SMILES.
    """
    if len(smiles1) < 2 or len(smiles2) < 2: return smiles1
    cut1 = random.randint(1, len(smiles1)-1)
    cut2 = random.randint(1, len(smiles2)-1)
    return smiles1[:cut1] + smiles2[cut2:]

def hho_discrete(
    obj_func: Callable[[str], float],
    initial_population: List[str],
    max_iter: int = 50
) -> Tuple[str, float, List[float]]:
    """
    Version discrète de HHO adaptée aux chaînes de caractères (SMILES).
    """
    pop_size = len(initial_population)
    X = initial_population.copy()
    
    rabbit_location = ""
    rabbit_energy = float("inf")
    
    convergence_curve = []
    
    for t in range(max_iter):
        for i in range(pop_size):
            fitness = obj_func(X[i])
            if fitness < rabbit_energy:
                rabbit_energy = fitness
                rabbit_location = X[i]
                
        E1 = 2 * (1 - (t / max_iter))
        
        for i in range(pop_size):
            E0 = 2 * random.random() - 1
            E = E1 * E0
            
            if abs(E) >= 1:
                # Phase d'exploration
                q = random.random()
                rand_hawk_index = random.randint(0, pop_size - 1)
                X_rand = X[rand_hawk_index]
                
                if q >= 0.5:
                    X[i] = mutate_smiles(crossover_smiles(X[i], X_rand), intensity=0.3)
                else:
                    X[i] = mutate_smiles(X[i], intensity=0.8)
            else:
                # Phase d'exploitation
                r = random.random()
                
                if r >= 0.5 and abs(E) >= 0.5:
                    X[i] = mutate_smiles(crossover_smiles(X[i], rabbit_location), intensity=0.1)
                elif r >= 0.5 and abs(E) < 0.5:
                    X[i] = mutate_smiles(rabbit_location, intensity=0.1)
                elif r < 0.5 and abs(E) >= 0.5:
                    Y = crossover_smiles(X[i], rabbit_location)
                    if obj_func(Y) < obj_func(X[i]):
                        X[i] = Y
                    else:
                        Z = mutate_smiles(Y, intensity=0.3)
                        if obj_func(Z) < obj_func(X[i]):
                            X[i] = Z
                elif r < 0.5 and abs(E) < 0.5:
                    Y = mutate_smiles(rabbit_location, intensity=0.2)
                    if obj_func(Y) < obj_func(X[i]):
                        X[i] = Y
                    else:
                        Z = mutate_smiles(Y, intensity=0.4)
                        if obj_func(Z) < obj_func(X[i]):
                            X[i] = Z

        convergence_curve.append(rabbit_energy)
        
    return rabbit_location, rabbit_energy, convergence_curve

# --- Exécution du HHO Discret ---
initial_smiles_pop = [
    "CCO", "CC(=O)O", "CCCC", "c1ccccc1", "CCN",
    "C1=CC=CC=C1", "CC(C)C", "C=C", "CC#N", "C1CCCCC1",
    "C(C(=O)O)N", "c1cc(O)ccc1", "C1=CN=CN1", "C1=CC=NC=C1", "C1COCCO1",
    "CC1=CC=CC=C1", "CC(=O)C", "CC(O)C", "C1=CC=C(C=C1)O", "C1=CC=C(C=C1)N"
]

best_smiles, best_score, conv_curve_smiles = hho_discrete(
    obj_func=smiles_fitness, 
    initial_population=initial_smiles_pop, 
    max_iter=50
)

print("Meilleure molécule trouvée :", best_smiles)
mol = Chem.MolFromSmiles(best_smiles)
if mol:
    print(f"Poids Moléculaire : {Descriptors.MolWt(mol):.2f}")
    print(f"Donneurs H : {Descriptors.NumHDonors(mol)}")
    print(f"Accepteurs H : {Descriptors.NumHAcceptors(mol)}")
print(f"Score (à minimiser) : {best_score:.2f}")

plt.figure(figsize=(8, 5))
plt.plot(conv_curve_smiles, linewidth=2, color='g')
plt.title('Convergence HHO Discret - Drug Design (SMILES)')
plt.xlabel('Itérations')
plt.ylabel('Meilleur Score (Fitness)')
plt.grid(True)
plt.show()

if mol:
    from rdkit.Chem import Draw
    img = Draw.MolToImage(mol)
    display(img)

In [ ]:
# --- Cas 2 : Visualisation sur une fonction de test 2D (Fonction de Sphere) ---

def sphere_function(x: np.ndarray) -> float:
    """
    Fonction de Sphere pour tester l'optimisation (minimum global à x=0, y=0).
    """
    return np.sum(x**2)

lb_sphere = np.array([-5, -5])
ub_sphere = np.array([5, 5])

best_pos, best_val, conv_curve_sphere, history = hho(
    obj_func=sphere_function, 
    lb=lb_sphere, 
    ub=ub_sphere, 
    dim=2, 
    pop_size=30, 
    max_iter=50
)

# Visualisation de l'espace de recherche et de l'historique
x_vals = np.linspace(-5, 5, 100)
y_vals = np.linspace(-5, 5, 100)
X_grid, Y_grid = np.meshgrid(x_vals, y_vals)
Z_grid = X_grid**2 + Y_grid**2

history_x = [pos[0] for pos in history]
history_y = [pos[1] for pos in history]

plt.figure(figsize=(10, 6))
plt.contourf(X_grid, Y_grid, Z_grid, levels=50, cmap='viridis', alpha=0.8)
plt.colorbar(label='Valeur de la fonction')
plt.plot(history_x, history_y, marker='o', color='red', linestyle='-', markersize=4, label='Trajectoire du meilleur Faucon')
plt.plot(0, 0, marker='*', color='gold', markersize=15, label='Optimum global')

plt.title('Visualisation de l\'algorithme HHO sur la fonction Sphere 2D')
plt.xlabel('X1')
plt.ylabel('X2')
plt.legend()
plt.grid(True)
plt.show()